In [1]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import classification_report
import nltk
import numpy as np
nltk.download('punkt')
from nltk.tokenize import sent_tokenize
nltk.download('punkt_tab')
from transformers.pipelines.pt_utils import KeyDataset
import argparse, json, re, sys
from typing import List, Optional
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer



[nltk_data] Downloading package punkt to
[nltk_data]     /ihome/xli/sek188/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /ihome/xli/sek188/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
def load_model(model_name: str, load_in_4bit: bool):
    if load_in_4bit:
        try:
            from transformers import BitsAndBytesConfig
            bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_use_double_quant=True,
                                            bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                device_map="auto",
                quantization_config=bnb_config,
                trust_remote_code=True,
            )
        except Exception as e:
            print(f"[warn] 4-bit load failed ({e}); falling back to fp16.", file=sys.stderr)
            model = AutoModelForCausalLM.from_pretrained(
                model_name,
                device_map="auto",
                torch_dtype=torch.float16,
                trust_remote_code=True,
            )
    else:
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            trust_remote_code=True,
        )
    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return tok, model

In [ ]:
def build_messages(sys_prompt: str, instruction: str, text: str, labels: List[str]) -> List[dict]:
    # Constrain the model to emit only a JSON object with one of the allowed labels.
    label_list_str = ", ".join([f'"{x}"' for x in labels])
    user_content = (
        f"{instruction}\n\n"
        f"TEXT:\n{text}\n\n"
        f"Respond ONLY with JSON of the form: {{\"label\": one of [{label_list_str}]}}"
    )
    return [
        {"role": "system", "content": sys_prompt},
        {"role": "user", "content": user_content},
    ]

JSON_RE = re.compile(r"\{[^{}]*\}")

In [ ]:
def parse_label(raw: str, labels: List[str]) -> Optional[str]:
    # Try JSON first
    m = JSON_RE.search(raw)
    if m:
        try:
            obj = json.loads(m.group(0))
            lab = obj.get("label")
            if isinstance(lab, str) and lab in labels:
                return lab
        except Exception:
            pass
    # Fallback: look for any allowed label as a standalone token
    for lab in labels:
        if re.search(rf"(?:^|\b){re.escape(lab)}(?:\b|$)", raw):
            return lab
    return None

In [ ]:
#!/usr/bin/env python3
@torch.inference_mode()
def generate_labels_batch(
    tok, model, rows: List[str], sys_prompt: str, instruction: str,
    labels: List[str], max_new_tokens: int, temperature: float, top_p: float
) -> List[str]:
    messages_list = [build_messages(sys_prompt, instruction, t, labels) for t in rows]
    texts = [tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in messages_list]
    enc = tok(texts, return_tensors="pt", padding=True, truncation=True).to(model.device)

    gen = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=(temperature > 0),
        temperature=temperature,
        top_p=top_p,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.pad_token_id,
    )
    # Slice off the prompt for each sequence
    outs = []
    for i in range(gen.size(0)):
        prompt_len = enc["input_ids"][i].size(0)
        out_ids = gen[i][prompt_len:]
        text = tok.decode(out_ids, skip_special_tokens=True)
        lab = parse_label(text, labels) or "UNPARSED"
        outs.append(lab)
    return outs

def main():
    ap = argparse.ArgumentParser(description="Label a DataFrame (CSV) with Qwen using a custom prompt.")
    ap.add_argument("--infile", required=True, help="Input CSV with a 'clean' column.")
    ap.add_argument("--outfile", required=True, help="Output CSV with added label column.")
    ap.add_argument("--text-col", default="clean", help="Column containing the text (default: clean).")
    ap.add_argument("--out-col", default="qwen_label", help="Name of output column (default: qwen_label).")
    ap.add_argument("--model", default="Qwen/Qwen2.5-0.5B-Instruct", help="HF model id.")
    ap.add_argument("--labels", default="LABEL_0,LABEL_1", help="Comma-separated allowed labels.")
    ap.add_argument("--system-prompt", default="You are a careful classifier. Always follow the instructions exactly.",
                    help="System message.")
    ap.add_argument("--instruction", required=True,
                    help="User instruction describing how to assign a label (must return one of the allowed labels).")
    ap.add_argument("--batch-size", type=int, default=16, help="Batch size for generation.")
    ap.add_argument("--max-new-tokens", type=int, default=16)
    ap.add_argument("--temperature", type=float, default=0.0)
    ap.add_argument("--top-p", type=float, default=1.0)
    ap.add_argument("--load-in-4bit", action="store_true", help="Try to load the model in 4-bit.")
    args = ap.parse_args()

    labels = [x.strip() for x in args.labels.split(",") if x.strip()]
    if not labels:
        raise ValueError("No labels provided.")

    print(f"[info] loading model: {args.model}", file=sys.stderr)
    tok, model = load_model(args.model, args.load_in_4bit)

    print(f"[info] reading {args.infile}", file=sys.stderr)
    df = pd.read_csv(args.infile)
    if args.text_col not in df.columns:
        raise ValueError(f"Column '{args.text_col}' not found in CSV.")

    texts = df[args.text_col].fillna("").astype(str).tolist()
    preds = []

    bs = max(1, args.batch_size)
    for i in range(0, len(texts), bs):
        batch = texts[i:i+bs]
        labs = generate_labels_batch(
            tok, model, batch, args.system_prompt, args.instruction,
            labels, args.max_new_tokens, args.temperature, args.top_p
        )
        preds.extend(labs)
        print(f"[info] processed {min(i+bs, len(texts))}/{len(texts)}", file=sys.stderr)

    df[args.out_col] = preds
    df.to_csv(args.outfile, index=False)
    print(f"[done] wrote {args.outfile}", file=sys.stderr)

if __name__ == "__main__":
    main()
